# MLB Sticky Stuff — SQL Analysis
### BUSN30135 — Alex Casarella, Corey Savedoff, Jack Kohr

This notebook contains all required SQL queries for the final project. All queries run against the SQLite database `mlb_sticky_stuff.sqlite`, which contains three tables:

| Table | Description | Key Columns |
|-------|-------------|-------------|
| `pitching_stats` | Season-level traditional pitching stats from the MLB Stats API | player_id, player_name, season, era, K_pct, earned_run_avg, WHIP, BAA, HR9, IP, had_significant_injury |
| `statcast_agg` | Aggregated Statcast pitch data (spin rate, velocity, movement) per pitcher-season | player_id, season, avg_spin_rate, avg_velocity, avg_hmov, avg_vmov, pct_fastball |
| `injuries` | Injured list (IL) transactions from the MLB API | player_id, player_name, season, description, is_60_day |

The database was created in the main analysis notebook by loading DataFrames via `df.to_sql()`.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('/Users/alexcasarella/Downloads/mlb_sticky_stuff.sqlite')

# Verify tables exist
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"Tables in database: {tables['name'].tolist()}")

# Quick row counts
for tbl in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {tbl}", conn)
    print(f"  {tbl}: {count['n'].values[0]} rows")

Tables in database: ['statcast_agg', 'injuries', 'pitching_stats']
  statcast_agg: 2157 rows
  injuries: 8907 rows
  pitching_stats: 1464 rows


---
## Query 1: JOIN — Pitching Stats with Statcast Spin Rates

**WHAT:** Combines traditional pitching statistics with Statcast spin rate and velocity data for each pitcher-season.

**HOW:** Uses a LEFT JOIN between `pitching_stats` and `statcast_agg` on player_id and season to attach spin rate data to each pitcher's seasonal record.

**WHY:** This is the foundational query for our analysis — linking downstream performance stats (K%, ERA) with the direct mechanism (spin rate). If foreign substances were boosting spin, we should see that pitchers with higher pre-crackdown spin rates experienced larger K% declines post-crackdown.

**OUTPUT:** Each row shows a pitcher-season with both traditional stats and Statcast metrics side by side. Filtered to pre/post eras only (excluding COVID 2020 and transition 2021).

In [2]:
# Query 1: JOIN — Pitching stats with Statcast spin rates
q1 = """
SELECT p.player_name, p.season, p.era, p.K_pct, p.earned_run_avg, p.WHIP,
       s.avg_spin_rate, s.avg_velocity, s.total_pitches
FROM pitching_stats p
LEFT JOIN statcast_agg s 
    ON p.player_id = s.player_id AND p.season = s.season
WHERE p.era_binary IS NOT NULL
ORDER BY p.player_name, p.season
LIMIT 20;
"""
display(pd.read_sql(q1, conn))

,player_name,season,era,K_pct,earned_run_avg,WHIP,avg_spin_rate,avg_velocity,total_pitches
0,A.J. Minter,2022,post_crackdown,0.347,2.06,0.91,2343.173310,93.018631,1154
1,A.J. Minter,2023,post_crackdown,0.315,3.76,1.19,2368.866364,92.134330,1100
2,A.J. Puk,2022,post_crackdown,0.270,3.12,1.15,2246.197946,92.889179,1071
3,A.J. Puk,2023,post_crackdown,0.322,3.97,1.18,2238.985030,90.679146,1002
4,Aaron Ashby,2022,post_crackdown,0.265,4.44,1.43,2225.374200,88.720501,1876
5,Aaron Brooks,2019,pre_crackdown,0.170,5.65,1.38,2059.819381,88.588592,1971
6,Aaron Bummer,2019,pre_crackdown,0.229,2.13,0.99,2075.312000,93.640616,1000
7,Aaron Bummer,2023,post_crackdown,0.292,6.79,1.53,2258.111010,89.074704,1099
8,Aaron Civale,2019,pre_crackdown,0.203,2.34,1.04,2501.213497,87.468177,815
9,Aaron Civale,2022,post_crackdown,0.241,4.92,1.19,2600.580134,84.753874,1641


---
## Query 2: JOIN — Pitching Stats with Injury Details (2021)

**WHAT:** Links pitching performance in 2021 (the crackdown year) with specific injury list transactions for each pitcher.

**HOW:** LEFT JOIN between `pitching_stats` and `injuries` on player_id, filtered to the 2021 season. This shows which pitchers on the IL that year also had qualifying performance data.

**WHY:** The 2021 season is critical because the crackdown happened mid-season. By joining injury data, we can identify pitchers like Tyler Glasnow whose performance drops were injury-related rather than crackdown-related. This separation is essential for honest analysis.

**OUTPUT:** Pitcher name, 2021 stats, injury flag, and injury description (NULL if no IL stint).

In [3]:
# Query 2: JOIN — Pitching stats with injury details for 2021
q2 = """
SELECT DISTINCT p.player_name, p.season, 
       ROUND(p.K_pct, 4) AS K_pct, 
       ROUND(p.earned_run_avg, 2) AS earned_run_avg,
       p.had_significant_injury,
       i.description AS injury_description
FROM pitching_stats p
LEFT JOIN injuries i 
    ON p.player_id = i.player_id AND p.season = i.season
WHERE p.season = 2021
ORDER BY p.K_pct DESC
LIMIT 20;
"""
display(pd.read_sql(q2, conn))

,player_name,season,K_pct,earned_run_avg,had_significant_injury,injury_description
0,Josh Hader,2021,0.455,1.23,0,Milwaukee Brewers activated LHP Josh Hader fro...
1,Josh Hader,2021,0.455,1.23,0,Milwaukee Brewers placed LHP Josh Hader on the...
2,Jacob deGrom,2021,0.451,1.08,1,New York Mets activated RHP Jacob deGrom from ...
3,Jacob deGrom,2021,0.451,1.08,1,New York Mets placed RHP Jacob deGrom on the 1...
4,Jacob deGrom,2021,0.451,1.08,1,New York Mets placed RHP Jacob deGrom on the 1...
5,Jacob deGrom,2021,0.451,1.08,1,New York Mets placed RHP Jacob deGrom on the 6...
6,Craig Kimbrel,2021,0.426,2.26,0,None
7,Liam Hendriks,2021,0.423,2.54,0,None
8,Aroldis Chapman,2021,0.399,3.36,0,New York Yankees activated LHP Aroldis Chapman...
9,Aroldis Chapman,2021,0.399,3.36,0,New York Yankees placed LHP Aroldis Chapman on...


---
## Query 3: JOIN + Subquery — High-Spin Pitchers and Their K% Change

**WHAT:** Identifies pitchers who had above-average spin rates in 2019 (pre-crackdown) and shows how their K% changed from pre to post crackdown.

**HOW:** Uses a subquery `(SELECT AVG(avg_spin_rate) FROM statcast_agg WHERE season = 2019)` to compute the league-average spin rate, then JOINs pre-crackdown and post-crackdown pitching stats with Statcast data, filtering to pitchers above that average.

**WHY:** This directly tests our core hypothesis: if sticky substances were boosting spin rates, then pitchers with the highest pre-crackdown spin rates should show the largest K% declines. This query isolates those high-spin pitchers.

**OUTPUT:** Pitcher name, pre/post K%, the K% decline, and their 2019 spin rate. Sorted by largest decline.

In [4]:
# Query 3: JOIN + Subquery — High-spin pitchers with K% change
q3 = """
SELECT pre.player_name, 
       ROUND(pre.K_pct, 4) AS k_pct_pre, 
       ROUND(post_avg.avg_k_pct, 4) AS k_pct_post,
       ROUND(post_avg.avg_k_pct - pre.K_pct, 4) AS k_pct_change,
       ROUND(sc.avg_spin_rate, 0) AS spin_rate_2019
FROM pitching_stats pre
JOIN (
    SELECT player_id, AVG(K_pct) AS avg_k_pct 
    FROM pitching_stats 
    WHERE era = 'post_crackdown' 
    GROUP BY player_id
) post_avg ON pre.player_id = post_avg.player_id
JOIN statcast_agg sc 
    ON pre.player_id = sc.player_id AND sc.season = 2019
WHERE pre.era = 'pre_crackdown'
  AND sc.avg_spin_rate > (SELECT AVG(avg_spin_rate) FROM statcast_agg WHERE season = 2019)
ORDER BY k_pct_change ASC
LIMIT 20;
"""
display(pd.read_sql(q3, conn))

,player_name,k_pct_pre,k_pct_post,k_pct_change,spin_rate_2019
0,Mike Clevinger,0.339,0.1940,-0.1450,2289.0
1,Patrick Corbin,0.285,0.1685,-0.1165,2254.0
2,Justin Verlander,0.354,0.2465,-0.1075,2598.0
3,Gerrit Cole,0.399,0.2970,-0.1020,2566.0
4,John Brebbia,0.286,0.1880,-0.0980,2482.0
5,Rich Hill,0.298,0.2010,-0.0970,2665.0
6,Mark Melancon,0.239,0.1420,-0.0970,2450.0
7,Brad Hand,0.347,0.2500,-0.0970,2444.0
8,Colin Poche,0.348,0.2545,-0.0935,2296.0
9,Seth Lugo,0.331,0.2430,-0.0880,2473.0


---
## Query 4: Window Function (RANK) — Rank Pitchers by Spin Rate Within Each Era

**WHAT:** Ranks all pitchers by their average spin rate within each era (pre-crackdown vs. post-crackdown).

**HOW:** Uses `RANK() OVER(PARTITION BY p.era ORDER BY s.avg_spin_rate DESC)` to assign a spin rate rank within each era. The PARTITION BY ensures rankings restart for each era.

**WHY:** This shows whether the same pitchers dominate spin rate rankings in both eras. If sticky substances were responsible for high spin rates, we might see different pitchers at the top pre vs. post crackdown, as those who relied on substances would drop in the rankings.

**OUTPUT:** Top 10 spin rate pitchers in each era with their rank, spin rate, and K%.

In [5]:
# Query 4: Window (RANK) — Rank pitchers by spin rate within each era
q4 = """
SELECT * FROM (
    SELECT p.player_name, p.era, 
           ROUND(s.avg_spin_rate, 0) AS avg_spin_rate,
           ROUND(p.K_pct, 4) AS K_pct,
           RANK() OVER(PARTITION BY p.era ORDER BY s.avg_spin_rate DESC) AS spin_rank
    FROM pitching_stats p
    JOIN statcast_agg s 
        ON p.player_id = s.player_id AND p.season = s.season
    WHERE p.era_binary IS NOT NULL
)
WHERE spin_rank <= 10
ORDER BY era, spin_rank;
"""
display(pd.read_sql(q4, conn))

,player_name,era,avg_spin_rate,K_pct,spin_rank
0,Lucas Sims,post_crackdown,2844.0,0.279,1
1,Phil Maton,post_crackdown,2820.0,0.270,2
2,Chris Stratton,post_crackdown,2805.0,0.215,3
3,Ryan Pressly,post_crackdown,2780.0,0.276,4
4,Clarke Schmidt,post_crackdown,2771.0,0.215,5
5,Pierce Johnson,post_crackdown,2709.0,0.325,6
6,Chris Stratton,post_crackdown,2704.0,0.240,7
7,David Robertson,post_crackdown,2692.0,0.290,8
8,Bryan Abreu,post_crackdown,2691.0,0.348,9
9,Alexis Díaz,post_crackdown,2688.0,0.325,10


---
## Query 5: Window Function (LAG) — Year-over-Year Spin Rate Changes

**WHAT:** Calculates each pitcher's year-over-year change in average spin rate by comparing the current season to the previous season.

**HOW:** Uses `LAG(s.avg_spin_rate, 1) OVER(PARTITION BY p.player_id ORDER BY p.season)` to retrieve each pitcher's spin rate from the prior season, then computes the difference.

**WHY:** This reveals *when* each pitcher's spin rate dropped — was it a gradual decline, or did it happen suddenly between the 2021 and 2022 seasons (when the crackdown was in full effect)? A sudden drop concentrated around 2021-2022 would be strong evidence of a crackdown effect.

**OUTPUT:** Pitcher name, season, current spin rate, previous season's spin rate, and the change. Sorted by largest decline.

In [6]:
# Query 5: Window (LAG) — Year-over-year spin rate changes
q5 = """
SELECT player_name, season, 
       ROUND(avg_spin_rate, 0) AS spin_rate,
       ROUND(prev_spin, 0) AS prev_season_spin,
       ROUND(spin_change, 0) AS spin_change
FROM (
    SELECT p.player_name, p.player_id, p.season, s.avg_spin_rate,
           LAG(s.avg_spin_rate, 1) OVER(
               PARTITION BY p.player_id ORDER BY p.season
           ) AS prev_spin,
           s.avg_spin_rate - LAG(s.avg_spin_rate, 1) OVER(
               PARTITION BY p.player_id ORDER BY p.season
           ) AS spin_change
    FROM pitching_stats p
    JOIN statcast_agg s 
        ON p.player_id = s.player_id AND p.season = s.season
)
WHERE spin_change IS NOT NULL
ORDER BY spin_change ASC
LIMIT 20;
"""
display(pd.read_sql(q5, conn))

,player_name,season,spin_rate,prev_season_spin,spin_change
0,Aroldis Chapman,2021,2175.0,2492.0,-317.0
1,Carlos Carrasco,2021,2034.0,2322.0,-288.0
2,Shawn Armstrong,2022,2311.0,2584.0,-273.0
3,Jose Alvarez,2021,2002.0,2264.0,-261.0
4,Logan Gilbert,2023,1884.0,2139.0,-255.0
5,Erik Swanson,2023,1751.0,2003.0,-252.0
6,Dylan Bundy,2022,2145.0,2369.0,-224.0
7,Marcus Stroman,2021,2324.0,2543.0,-220.0
8,Tyler Beede,2022,1965.0,2168.0,-203.0
9,Emilio Pagán,2022,2241.0,2445.0,-203.0


---
## Query 6: Window Function (NTILE) + Subquery — K% Decline Quartiles

**WHAT:** Divides all qualified pitchers into quartiles based on their K% change from pre to post crackdown.

**HOW:** Uses a subquery to compute each pitcher's K% change (post average minus pre), then applies `NTILE(4) OVER(ORDER BY K_pct_change)` to assign each pitcher to a quartile. Quartile 1 contains the pitchers with the largest declines.

**WHY:** This creates the natural groupings we use in our logistic regression target variable. Pitchers in quartile 1 are the "most affected" — these are the pitchers we hypothesize were most reliant on foreign substances. Comparing average spin rates across quartiles tests whether spin loss correlates with K% decline severity.

**OUTPUT:** Pitcher name, pre/post K%, K% change, spin rate, and their decline quartile (1 = most declined).

In [7]:
# Query 6: Window (NTILE) + Subquery — K% decline quartiles
q6 = """
SELECT player_name, 
       ROUND(K_pct_pre, 4) AS K_pct_pre, 
       ROUND(K_pct_post, 4) AS K_pct_post,
       ROUND(K_pct_change, 4) AS K_pct_change,
       ROUND(spin_rate_2019, 0) AS spin_rate_2019,
       decline_quartile
FROM (
    SELECT pre.player_name, pre.player_id,
           pre.K_pct AS K_pct_pre, 
           post_avg.avg_k_pct AS K_pct_post,
           post_avg.avg_k_pct - pre.K_pct AS K_pct_change,
           sc.avg_spin_rate AS spin_rate_2019,
           NTILE(4) OVER(ORDER BY post_avg.avg_k_pct - pre.K_pct) AS decline_quartile
    FROM pitching_stats pre
    JOIN (
        SELECT player_id, AVG(K_pct) AS avg_k_pct 
        FROM pitching_stats 
        WHERE era = 'post_crackdown' 
        GROUP BY player_id
    ) post_avg ON pre.player_id = post_avg.player_id
    LEFT JOIN statcast_agg sc 
        ON pre.player_id = sc.player_id AND sc.season = 2019
    WHERE pre.era = 'pre_crackdown'
)
ORDER BY decline_quartile, K_pct_change;
"""
display(pd.read_sql(q6, conn))

,player_name,K_pct_pre,K_pct_post,K_pct_change,spin_rate_2019,decline_quartile
0,Mike Clevinger,0.339,0.1940,-0.1450,2289.0,1
1,Will Smith,0.374,0.2460,-0.1280,2241.0,1
2,Patrick Corbin,0.285,0.1685,-0.1165,2254.0,1
3,Josh Hader,0.478,0.3690,-0.1090,2191.0,1
4,Justin Verlander,0.354,0.2465,-0.1075,2598.0,1
...,...,...,...,...,...,...
170,Erik Swanson,0.212,0.3130,0.1010,2172.0,4
171,Yusei Kikuchi,0.161,0.2660,0.1050,2172.0,4
172,Jacob deGrom,0.317,0.4270,0.1100,2302.0,4
173,Edwin Díaz,0.390,0.5020,0.1120,2370.0,4


---
## Query 7: GROUP BY with ROLLUP — Average Stats by Era and Injury Status

**WHAT:** Summarizes average K%, ERA, and spin rate grouped by era and injury status, with subtotals for each era and a grand total.

**HOW:** SQLite does not natively support ROLLUP syntax, so we simulate it using UNION ALL. The first SELECT groups by both era and injury status (detail rows). The second groups by era only (era subtotals). The third provides the grand total. This is functionally equivalent to `GROUP BY ROLLUP(era, had_significant_injury)` in PostgreSQL/MySQL.

**WHY:** ROLLUP gives us hierarchical subtotals that are useful for reporting. We can see how injury status affects performance within each era, how each era compares overall, and the grand average — all in one result set. This is exactly the kind of summary table an MLB commissioner would want.

**OUTPUT:** Crosstab with era × injury status detail rows, era subtotals (injury_status = 'ALL'), and grand total (both = 'ALL').

In [8]:
# Query 7: ROLLUP equivalent via UNION ALL — Avg stats by era and injury status
# Note: SQLite does not support GROUP BY ROLLUP natively.
# This UNION ALL approach produces the same hierarchical subtotals.
# In PostgreSQL/MySQL, this would be: GROUP BY ROLLUP(era, had_significant_injury)

q7 = """
-- Detail rows: era × injury status
SELECT era, 
       CAST(had_significant_injury AS TEXT) AS injury_status,
       COUNT(*) AS n_pitchers,
       ROUND(AVG(K_pct), 4) AS avg_k_pct,
       ROUND(AVG(earned_run_avg), 2) AS avg_era
FROM pitching_stats
WHERE era_binary IS NOT NULL
GROUP BY era, had_significant_injury

UNION ALL

-- Era subtotals (across all injury statuses)
SELECT era,
       'ALL' AS injury_status,
       COUNT(*) AS n_pitchers,
       ROUND(AVG(K_pct), 4) AS avg_k_pct,
       ROUND(AVG(earned_run_avg), 2) AS avg_era
FROM pitching_stats
WHERE era_binary IS NOT NULL
GROUP BY era

UNION ALL

-- Grand total
SELECT 'ALL ERAS' AS era,
       'ALL' AS injury_status,
       COUNT(*) AS n_pitchers,
       ROUND(AVG(K_pct), 4) AS avg_k_pct,
       ROUND(AVG(earned_run_avg), 2) AS avg_era
FROM pitching_stats
WHERE era_binary IS NOT NULL

ORDER BY era, injury_status;
"""
display(pd.read_sql(q7, conn))

,era,injury_status,n_pitchers,avg_k_pct,avg_era
0,ALL ERAS,ALL,1045,0.2364,4.06
1,post_crackdown,0,534,0.2353,3.94
2,post_crackdown,1,170,0.2366,3.96
3,post_crackdown,ALL,704,0.2356,3.95
4,pre_crackdown,0,226,0.2346,4.39
5,pre_crackdown,1,115,0.2450,4.13
6,pre_crackdown,ALL,341,0.2381,4.30


---
## Query 8: GROUP BY with CUBE — Stats by Era and Pitch Profile

**WHAT:** Cross-tabulates era and pitch profile (fastball-heavy vs. offspeed-heavy) with all possible subtotal combinations.

**HOW:** SQLite does not natively support CUBE, so we simulate it with UNION ALL. A true CUBE on two dimensions produces 2² = 4 grouping combinations: (era, profile), (era, ALL), (ALL, profile), (ALL, ALL). We compute each combination separately and union them.

**WHY:** CUBE goes beyond ROLLUP by providing subtotals in *every* dimension. This tells us whether fastball-heavy pitchers were affected differently than offspeed-heavy pitchers, regardless of era, and whether either era shows a pattern regardless of pitch profile. A scout would use this to understand how pitch selection interacts with the crackdown's effects.

**OUTPUT:** All 2² combinations of era × pitch profile with pitcher counts, average K%, and average spin rate.

In [9]:
# Query 8: CUBE equivalent via UNION ALL — Stats by era and pitch profile
# Note: SQLite does not support GROUP BY CUBE natively.
# A CUBE on (era, pitch_profile) produces 4 grouping sets.
# In PostgreSQL/MySQL: GROUP BY CUBE(era, pitch_profile)

q8 = """
-- Detail: era × pitch profile
SELECT p.era,
       CASE WHEN s.pct_fastball > 0.5 THEN 'fastball_heavy' ELSE 'offspeed_heavy' END AS pitch_profile,
       COUNT(*) AS n,
       ROUND(AVG(p.K_pct), 4) AS avg_k_pct,
       ROUND(AVG(s.avg_spin_rate), 0) AS avg_spin
FROM pitching_stats p
JOIN statcast_agg s ON p.player_id = s.player_id AND p.season = s.season
WHERE p.era_binary IS NOT NULL
GROUP BY p.era, CASE WHEN s.pct_fastball > 0.5 THEN 'fastball_heavy' ELSE 'offspeed_heavy' END

UNION ALL

-- Era subtotals (across all pitch profiles)
SELECT p.era,
       'ALL' AS pitch_profile,
       COUNT(*) AS n,
       ROUND(AVG(p.K_pct), 4) AS avg_k_pct,
       ROUND(AVG(s.avg_spin_rate), 0) AS avg_spin
FROM pitching_stats p
JOIN statcast_agg s ON p.player_id = s.player_id AND p.season = s.season
WHERE p.era_binary IS NOT NULL
GROUP BY p.era

UNION ALL

-- Pitch profile subtotals (across all eras)
SELECT 'ALL' AS era,
       CASE WHEN s.pct_fastball > 0.5 THEN 'fastball_heavy' ELSE 'offspeed_heavy' END AS pitch_profile,
       COUNT(*) AS n,
       ROUND(AVG(p.K_pct), 4) AS avg_k_pct,
       ROUND(AVG(s.avg_spin_rate), 0) AS avg_spin
FROM pitching_stats p
JOIN statcast_agg s ON p.player_id = s.player_id AND p.season = s.season
WHERE p.era_binary IS NOT NULL
GROUP BY CASE WHEN s.pct_fastball > 0.5 THEN 'fastball_heavy' ELSE 'offspeed_heavy' END

UNION ALL

-- Grand total
SELECT 'ALL' AS era,
       'ALL' AS pitch_profile,
       COUNT(*) AS n,
       ROUND(AVG(p.K_pct), 4) AS avg_k_pct,
       ROUND(AVG(s.avg_spin_rate), 0) AS avg_spin
FROM pitching_stats p
JOIN statcast_agg s ON p.player_id = s.player_id AND p.season = s.season
WHERE p.era_binary IS NOT NULL

ORDER BY era, pitch_profile;
"""
display(pd.read_sql(q8, conn))

,era,pitch_profile,n,avg_k_pct,avg_spin
0,ALL,ALL,1045,0.2364,2252.0
1,ALL,fastball_heavy,689,0.2322,2240.0
2,ALL,offspeed_heavy,356,0.2447,2275.0
3,post_crackdown,ALL,704,0.2356,2253.0
4,post_crackdown,fastball_heavy,439,0.2318,2238.0
5,post_crackdown,offspeed_heavy,265,0.2420,2277.0
6,pre_crackdown,ALL,341,0.2381,2250.0
7,pre_crackdown,fastball_heavy,250,0.2329,2244.0
8,pre_crackdown,offspeed_heavy,91,0.2524,2268.0


---
## Query 9: Subquery — Above-Average Spin Pitchers Ranked by K% Decline

**WHAT:** Filters to pitchers with above-league-average spin rates and ranks them by their K% from highest to lowest within the pre-crackdown era.

**HOW:** Uses a subquery `(SELECT AVG(avg_spin_rate) FROM statcast_agg)` in the WHERE clause to filter, and wraps the result in an outer query that filters to the top 20 by rank.

**WHY:** This identifies the elite spin-rate pitchers and their pre-crackdown dominance. Combined with Query 3 (which shows their post-crackdown K% change), this gives us a full picture of who was on top before enforcement and how far they fell.

**OUTPUT:** Top 20 above-average-spin pitchers by K% rank in the pre-crackdown era.

In [10]:
# Query 9: Subquery — Above-average spin pitchers ranked by K%
q9 = """
SELECT * FROM (
    SELECT p.player_name, 
           ROUND(s.avg_spin_rate, 0) AS spin_rate,
           ROUND(p.K_pct, 4) AS K_pct, 
           ROUND(p.earned_run_avg, 2) AS earned_run_avg,
           RANK() OVER(ORDER BY p.K_pct DESC) AS k_rank
    FROM pitching_stats p
    JOIN statcast_agg s 
        ON p.player_id = s.player_id AND p.season = s.season
    WHERE s.avg_spin_rate > (SELECT AVG(avg_spin_rate) FROM statcast_agg)
      AND p.era = 'pre_crackdown'
)
WHERE k_rank <= 20
ORDER BY k_rank;
"""
display(pd.read_sql(q9, conn))

,player_name,spin_rate,K_pct,earned_run_avg,k_rank
0,Nick Anderson,2244.0,0.417,3.32,1
1,Ken Giles,2284.0,0.399,1.87,2
2,Gerrit Cole,2566.0,0.399,2.50,2
3,Edwin Díaz,2370.0,0.390,5.59,4
4,Matt Barnes,2336.0,0.386,3.78,5
5,Felipe Vázquez,2609.0,0.381,1.65,6
6,Josh James,2373.0,0.376,4.70,7
7,Liam Hendriks,2310.0,0.373,1.80,8
8,Brandon Workman,2367.0,0.364,1.88,9
9,Aroldis Chapman,2492.0,0.362,2.21,10


---
## Query 10: Window Function (Running AVG) — Cumulative Average Spin Rate

**WHAT:** Computes a running (cumulative) average of spin rate across seasons for each pitcher.

**HOW:** Uses `AVG(s.avg_spin_rate) OVER(PARTITION BY p.player_id ORDER BY p.season ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)` to calculate the expanding average as each season is added.

**WHY:** The cumulative average smooths out year-to-year volatility and reveals whether a pitcher's spin rate trajectory was stable before the crackdown and then suddenly dropped. A pitcher with a stable 2300 RPM cumulative average that drops to 2200 after 2021 shows a clear structural break.

**OUTPUT:** Pitcher name, season, season spin rate, cumulative average spin rate, and K%.

In [11]:
# Query 10: Window (running AVG) — Cumulative average spin rate per pitcher
q10 = """
SELECT p.player_name, p.season, 
       ROUND(s.avg_spin_rate, 0) AS season_spin,
       ROUND(AVG(s.avg_spin_rate) OVER(
           PARTITION BY p.player_id 
           ORDER BY p.season 
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ), 0) AS cumulative_avg_spin,
       ROUND(p.K_pct, 4) AS K_pct
FROM pitching_stats p
JOIN statcast_agg s 
    ON p.player_id = s.player_id AND p.season = s.season
WHERE p.player_name IN ('Gerrit Cole', 'Tyler Glasnow', 'Max Scherzer', 'Jacob deGrom')
ORDER BY p.player_name, p.season;
"""
display(pd.read_sql(q10, conn))

,player_name,season,season_spin,cumulative_avg_spin,K_pct
0,Gerrit Cole,2019,2566.0,2566.0,0.399
1,Gerrit Cole,2021,2439.0,2503.0,0.335
2,Gerrit Cole,2022,2436.0,2480.0,0.324
3,Gerrit Cole,2023,2430.0,2468.0,0.270
4,Jacob deGrom,2019,2302.0,2302.0,0.317
5,Jacob deGrom,2021,2408.0,2355.0,0.451
6,Jacob deGrom,2022,2480.0,2397.0,0.427
7,Max Scherzer,2019,2344.0,2344.0,0.351
8,Max Scherzer,2021,2282.0,2313.0,0.341
9,Max Scherzer,2022,2278.0,2301.0,0.306


---
## Query 11: GROUP BY — Era Summary Statistics

**WHAT:** Provides a concise summary of all key metrics grouped by era (pre_crackdown, transition, post_crackdown, covid_excluded).

**HOW:** Standard GROUP BY with multiple aggregate functions (COUNT DISTINCT, AVG, ROUND).

**WHY:** This is the quick-reference table for the entire analysis. It shows at a glance how every metric changed across eras and provides the context for all subsequent deep-dive queries.

**OUTPUT:** One row per era with pitcher count, average K%, ERA, WHIP, BAA, and spin rate.

In [12]:
# Query 11: GROUP BY — Era summary statistics
q11 = """
SELECT p.era, 
       COUNT(DISTINCT p.player_id) AS n_pitchers,
       ROUND(AVG(p.K_pct), 4) AS avg_k_pct, 
       ROUND(AVG(p.earned_run_avg), 2) AS avg_era,
       ROUND(AVG(p.WHIP), 3) AS avg_whip, 
       ROUND(AVG(p.BAA), 4) AS avg_baa,
       ROUND(AVG(s.avg_spin_rate), 0) AS avg_spin_rate
FROM pitching_stats p
LEFT JOIN statcast_agg s 
    ON p.player_id = s.player_id AND p.season = s.season
GROUP BY p.era
ORDER BY MIN(p.season);
"""
display(pd.read_sql(q11, conn))

,era,n_pitchers,avg_k_pct,avg_era,avg_whip,avg_baa,avg_spin_rate
0,pre_crackdown,341,0.2381,4.30,1.296,0.2459,2250.0
1,covid_excluded,81,0.2406,3.85,1.219,0.2360,NaN
2,transition,338,0.2403,3.99,1.256,0.2358,2241.0
3,post_crackdown,488,0.2356,3.95,1.259,0.2394,2253.0


---
## Query 12: Window (ROW_NUMBER) + Subquery — Best Season per Pitcher per Era

**WHAT:** For pitchers who appear multiple times within an era (e.g., 2022 and 2023 in post_crackdown), keeps only their best K% season.

**HOW:** Uses `ROW_NUMBER() OVER(PARTITION BY player_id, era ORDER BY K_pct DESC)` to number each pitcher's seasons within an era by K% (best first), then the outer subquery filters to `rn = 1`.

**WHY:** This creates a clean one-row-per-pitcher-per-era dataset. For fair comparisons, we want each pitcher represented once per era. Choosing their best K% season is conservative — it means any decline we observe is likely understated.

**OUTPUT:** Each pitcher's single best season per era with K%, ERA, and row number.

In [13]:
# Query 12: Window (ROW_NUMBER) + Subquery — Deduplicate to best season per era
q12 = """
SELECT player_name, season, era, 
       ROUND(K_pct, 4) AS K_pct, 
       ROUND(earned_run_avg, 2) AS earned_run_avg
FROM (
    SELECT player_id, player_name, season, era, K_pct, earned_run_avg,
           ROW_NUMBER() OVER(
               PARTITION BY player_id, era 
               ORDER BY K_pct DESC
           ) AS rn
    FROM pitching_stats
    WHERE era_binary IS NOT NULL
)
WHERE rn = 1
ORDER BY player_name, era
LIMIT 20;
"""
display(pd.read_sql(q12, conn))

,player_name,season,era,K_pct,earned_run_avg
0,A.J. Minter,2022,post_crackdown,0.347,2.06
1,A.J. Puk,2023,post_crackdown,0.322,3.97
2,Aaron Ashby,2022,post_crackdown,0.265,4.44
3,Aaron Brooks,2019,pre_crackdown,0.170,5.65
4,Aaron Bummer,2023,post_crackdown,0.292,6.79
5,Aaron Bummer,2019,pre_crackdown,0.229,2.13
6,Aaron Civale,2022,post_crackdown,0.241,4.92
7,Aaron Civale,2019,pre_crackdown,0.203,2.34
8,Aaron Loup,2022,post_crackdown,0.200,3.84
9,Aaron Nola,2022,post_crackdown,0.291,3.25


---
## Query 13: Window (DENSE_RANK) + JOIN — Compare Velocity and Spin Rankings

**WHAT:** Ranks each pre-crackdown pitcher by both spin rate and velocity using DENSE_RANK, showing how the two rankings differ.

**HOW:** Uses two `DENSE_RANK() OVER(ORDER BY ...)` window functions — one for spin rate (descending) and one for velocity (descending). DENSE_RANK ensures no gaps in ranking for ties.

**WHY:** Velocity and spin rate are different physical attributes. Pitchers who rank much higher in spin than velocity may be artificially boosting spin through substances rather than raw arm strength. A large gap between spin_rank and velo_rank is suspicious.

**OUTPUT:** Top 15 pre-crackdown pitchers by spin rate with their spin rank, velocity rank, and the difference.

In [14]:
# Query 13: Window (DENSE_RANK) + JOIN — Compare velocity and spin rankings
q13 = """
SELECT player_name, 
       ROUND(avg_spin_rate, 0) AS spin_rate,
       ROUND(avg_velocity, 1) AS velocity,
       spin_rank, velo_rank,
       spin_rank - velo_rank AS rank_gap
FROM (
    SELECT p.player_name, s.avg_spin_rate, s.avg_velocity,
           DENSE_RANK() OVER(ORDER BY s.avg_spin_rate DESC) AS spin_rank,
           DENSE_RANK() OVER(ORDER BY s.avg_velocity DESC) AS velo_rank
    FROM pitching_stats p
    JOIN statcast_agg s 
        ON p.player_id = s.player_id AND p.season = s.season
    WHERE p.era = 'pre_crackdown'
)
ORDER BY spin_rank
LIMIT 15;
"""
display(pd.read_sql(q13, conn))

,player_name,spin_rate,velocity,spin_rank,velo_rank,rank_gap
0,Ryan Pressly,2879.0,89.4,1,139,-138
1,Chaz Roe,2790.0,84.7,2,319,-317
2,Sonny Gray,2676.0,88.4,3,202,-199
3,Rich Hill,2665.0,82.7,4,333,-329
4,Adam Ottavino,2647.0,87.7,5,231,-226
5,Chris Stratton,2636.0,87.9,6,221,-215
6,Will Harris,2633.0,87.1,7,256,-249
7,Yimi García,2622.0,89.2,8,151,-143
8,Felipe Vázquez,2609.0,93.4,9,14,-5
9,Walker Buehler,2608.0,92.7,10,28,-18


---
## Query 14: Correlated Subquery — Cole and Glasnow vs. League Average

**WHAT:** Shows Gerrit Cole's and Tyler Glasnow's K% each season alongside the league average K% for that same season, and the difference.

**HOW:** Uses a correlated subquery `(SELECT AVG(K_pct) FROM pitching_stats WHERE season = p.season)` that recalculates for each row based on the outer query's season value.

**WHY:** This contextualizes the two most-discussed crackdown cases. A pitcher's raw K% can be misleading without context — what matters is how far above (or below) league average they are. If Cole was 5% above average pre-crackdown but only 2% above post-crackdown, that tells a clearer story than raw numbers alone.

**OUTPUT:** Season-by-season K% for Cole and Glasnow, league average K%, and the difference (above/below average).

In [15]:
# Query 14: Correlated subquery — Cole & Glasnow vs league average
q14 = """
SELECT p.player_name, p.season, 
       ROUND(p.K_pct, 4) AS pitcher_k_pct,
       ROUND(
           (SELECT AVG(K_pct) FROM pitching_stats WHERE season = p.season), 
           4
       ) AS league_avg_k_pct,
       ROUND(
           p.K_pct - (SELECT AVG(K_pct) FROM pitching_stats WHERE season = p.season), 
           4
       ) AS k_pct_vs_avg
FROM pitching_stats p
WHERE p.player_name IN ('Gerrit Cole', 'Tyler Glasnow')
ORDER BY p.player_name, p.season;
"""
display(pd.read_sql(q14, conn))

,player_name,season,pitcher_k_pct,league_avg_k_pct,k_pct_vs_avg
0,Gerrit Cole,2019,0.399,0.2381,0.1609
1,Gerrit Cole,2020,0.326,0.2406,0.0854
2,Gerrit Cole,2021,0.335,0.2403,0.0947
3,Gerrit Cole,2022,0.324,0.2349,0.0891
4,Gerrit Cole,2023,0.270,0.2363,0.0337
5,Tyler Glasnow,2019,0.330,0.2381,0.0919
6,Tyler Glasnow,2020,0.382,0.2406,0.1414
7,Tyler Glasnow,2021,0.362,0.2403,0.1217
8,Tyler Glasnow,2023,0.334,0.2363,0.0977


---
## Query 15: Window Function (LEAD) — Next-Season Spin Rate Projections

**WHAT:** For each pitcher-season, shows what their spin rate and K% will be in the *next* season.

**HOW:** Uses `LEAD(s.avg_spin_rate, 1) OVER(PARTITION BY p.player_id ORDER BY p.season)` to look forward one season. LEAD is the complement of LAG — where LAG looks backward, LEAD looks forward.

**WHY:** This helps us see the trajectory of each pitcher going *into* the crackdown. A scout evaluating a pitcher in early 2021 would want to know if their spin rate was about to drop. By showing the next-season values alongside current values, we can identify which pitchers were on a collision course with enforcement.

**OUTPUT:** Pitcher name, current season stats, and next season's spin rate and K% (NULL if no next season data).

In [16]:
# Query 15: Window (LEAD) — Next-season spin rate and K%
q15 = """
SELECT player_name, season, 
       ROUND(spin_rate, 0) AS current_spin,
       ROUND(K_pct, 4) AS current_k_pct,
       ROUND(next_spin, 0) AS next_season_spin,
       ROUND(next_k_pct, 4) AS next_season_k_pct
FROM (
    SELECT p.player_name, p.player_id, p.season, 
           s.avg_spin_rate AS spin_rate, p.K_pct,
           LEAD(s.avg_spin_rate, 1) OVER(
               PARTITION BY p.player_id ORDER BY p.season
           ) AS next_spin,
           LEAD(p.K_pct, 1) OVER(
               PARTITION BY p.player_id ORDER BY p.season
           ) AS next_k_pct
    FROM pitching_stats p
    JOIN statcast_agg s 
        ON p.player_id = s.player_id AND p.season = s.season
)
WHERE next_spin IS NOT NULL
ORDER BY current_spin DESC
LIMIT 20;
"""
display(pd.read_sql(q15, conn))

,player_name,season,current_spin,current_k_pct,next_season_spin,next_season_k_pct
0,Ryan Pressly,2019,2879.0,0.341,2773.0,0.324
1,Chris Stratton,2022,2805.0,0.215,2704.0,0.240
2,Ryan Pressly,2021,2773.0,0.324,2780.0,0.276
3,Daniel Bard,2021,2760.0,0.263,2619.0,0.282
4,Chris Stratton,2021,2727.0,0.255,2805.0,0.215
5,Alexis Díaz,2022,2688.0,0.325,2598.0,0.301
6,Phil Maton,2021,2687.0,0.286,2664.0,0.260
7,Corbin Burnes,2021,2686.0,0.356,2601.0,0.305
8,Clarke Schmidt,2022,2679.0,0.237,2771.0,0.215
9,Sonny Gray,2019,2676.0,0.290,2486.0,0.270


---
## SQL Query Summary

| # | Query Type | SQL Techniques Used |
|---|-----------|--------------------|
| 1 | Pitching + Statcast | LEFT JOIN |
| 2 | Pitching + Injuries | LEFT JOIN |
| 3 | High-spin K% change | JOIN, Subquery (AVG filter), GROUP BY |
| 4 | Spin rank by era | JOIN, RANK() window, Subquery (filter) |
| 5 | YoY spin changes | JOIN, LAG() window, Subquery (filter) |
| 6 | K% decline quartiles | JOIN, NTILE() window, Subquery (aggregation) |
| 7 | Era × injury summary | GROUP BY, ROLLUP (via UNION ALL) |
| 8 | Era × pitch profile | JOIN, GROUP BY, CUBE (via UNION ALL) |
| 9 | High-spin K% leaders | JOIN, RANK() window, Subquery (AVG filter) |
| 10 | Cumulative avg spin | JOIN, Running AVG() window |
| 11 | Era summary | LEFT JOIN, GROUP BY, COUNT DISTINCT |
| 12 | Best season per era | ROW_NUMBER() window, Subquery (filter) |
| 13 | Spin vs velo rankings | JOIN, DENSE_RANK() window (×2) |
| 14 | Cole/Glasnow vs avg | Correlated subquery (×2) |
| 15 | Next-season projections | JOIN, LEAD() window |

**Rubric Coverage:**
- Total queries: **15** (requirement: 10+)
- JOINs: **13 queries** (requirement: 3+)
- Window functions: **8 queries** — RANK, LAG, NTILE, Running AVG, ROW_NUMBER, DENSE_RANK, LEAD (requirement: 2+)
- GROUP BY with ROLLUP/CUBE: **2 queries** — Q7 (ROLLUP), Q8 (CUBE) (requirement: 2+)
- Subqueries: **7 queries** — Q3, Q4, Q5, Q6, Q9, Q12, Q14 (requirement: 2+)

In [17]:
conn.close()
print("All queries executed successfully. Database connection closed.")

All queries executed successfully. Database connection closed.
